# 02 — Scale-in: добирать лонг частями вниз (зеркало pump-11)

Идея пользователя: не лонговать сразу целый ордер, а **добирать частями по мере
падения** → больше комиссий, но средний вход НИЖЕ (ближе ко дну) → ловим больше
отскока. Зеркало pump scale-in (там добирали шорт ВВЕРХ к пику).

Бьёт в проблему iter01: одиночный вход сидит через дип −3% до отскока; доборы
усредняют вход вниз во время этого дипа.

**Сетап (honest, lookahead-free):**
- Кластер: от первого триггера −X%/15m, пока условие не выключится на
  COOLDOWN=10 мин подряд.
- Транши каждые MIN_GAP=3 мин внутри кластера, рамп (поздние крупнее), Σ=1 поз.
- Вход транша = close[t+1]; выход = close[конец_кластера+240].
- Кости по числу филлов: 0.075%/сторона. single=2 филла; scale-in=N+1.
- Сравнение single vs scale при ОДИНАКОВОМ выходе → изолируем эффект входа.

In [1]:
import sys; sys.path.insert(0,".")
from _lab import *
import numpy as np
from pathlib import Path
z=np.load(Path("_out")/"closes_2024.npz",allow_pickle=True)
closes={k:z[k] for k in z.files}; SYMS=list(closes.keys())
WIN=15; H=240; COOLDOWN=10; MIN_GAP=3; N_MAX=10
FEE=0.00075  # per side

def clusters(c,thr):
    # returns list of (start_idx, end_idx, [tranche_minutes]) lookahead-free
    if len(c)<WIN+5: return []
    r=np.full(len(c),np.nan); r[WIN:]=c[WIN:]/c[:-WIN]-1
    cond=r<=-thr
    out=[]; i=WIN; n=len(c)
    while i<n:
        if not cond[i] or (i>0 and cond[i-1]):
            i+=1; continue
        # start of a cluster at i
        start=i; last_true=i; j=i; falses=0; tr=[i]; lastt=i
        j=i+1
        while j<n:
            if cond[j]:
                last_true=j; falses=0
                if j-lastt>=MIN_GAP and len(tr)<N_MAX:
                    tr.append(j); lastt=j
            else:
                falses+=1
                if falses>=COOLDOWN: break
            j+=1
        out.append((start,last_true,tr))
        i=last_true+COOLDOWN+1
    return out

In [2]:
# cluster length distribution (informs tranche count)
for thr in (0.05,0.07):
    L=[]; T=[]
    for s in SYMS:
        for (st,en,tr) in clusters(closes[s],thr):
            L.append(en-st); T.append(len(tr))
    L=np.array(L); T=np.array(T)
    print(f"-{int(thr*100)}%  n_clusters={len(L)}  cluster_len(min): med {np.median(L):.0f} "
          f"mean {L.mean():.1f} q90 {np.quantile(L,.9):.0f} | tranches: med {np.median(T):.0f} mean {T.mean():.1f} max {T.max()}")

-5%  n_clusters=1727  cluster_len(min): med 3 mean 6.1 q90 16 | tranches: med 2 mean 2.6 max 10


-7%  n_clusters=776  cluster_len(min): med 5 mean 6.2 q90 14 | tranches: med 2 mean 2.6 max 10


In [3]:
def stats(a):
    return dict(n=len(a),mean=a.mean(),median=np.median(a),win=(a>0).mean(),
                std=a.std(),q10=np.quantile(a,.10),ms=a.mean()/a.std() if a.std()>0 else np.nan)
def line(tag,a):
    s=stats(a)
    print(f"  {tag:28} mean {s['mean']*100:+6.2f}  med {s['median']*100:+6.2f}  "
          f"win {s['win']*100:4.1f}%  q10 {s['q10']*100:+6.2f}  m/std {s['ms']:+.3f}  (n={s['n']})")

def simulate(thr):
    single=[]; scaled=[]; entry_impr=[]; ntr=[]
    for s in SYMS:
        c=closes[s]; n=len(c)
        for (st,en,tr) in clusters(c,thr):
            ex_idx=en+1+H
            if ex_idx>=n or st+1>=n: continue
            exitp=c[ex_idx]
            # single-shot: full at first trigger+1
            e_single=c[st+1]
            pnl_s=exitp/e_single-1 - 2*FEE
            single.append(pnl_s)
            # scale-in: ramped weights over tranche fills (later bigger)
            ws=np.arange(1,len(tr)+1,dtype=float); ws/=ws.sum()
            prices=np.array([c[t+1] for t in tr if t+1<n])
            ws=ws[:len(prices)]; ws/=ws.sum()
            avg_entry=(ws*prices).sum()
            # fee is notional-based: splitting into N fills => entry fee still = FEE on
            # total notional (sum of per-fill FEE*weight), + exit FEE => 2*FEE total.
            pnl_sc=exitp/avg_entry-1 - 2*FEE
            scaled.append(pnl_sc)
            entry_impr.append(e_single/avg_entry-1)  # >0 => scaled entry lower
            ntr.append(len(prices))
    return map(np.array,(single,scaled,entry_impr,ntr))

for thr in (0.05,0.07):
    single,scaled,impr,ntr=simulate(thr)
    print(f"\n=== -{int(thr*100)}%/15m  (exit = cluster_end+{H}m, net fees) ===")
    line("single-shot (first trig)",single)
    line("scale-in (ramped down)",scaled)
    print(f"  avg tranches/cluster: {ntr.mean():.1f} | entry lowered by: med {np.median(impr)*100:+.2f}%  mean {impr.mean()*100:+.2f}%")


=== -5%/15m  (exit = cluster_end+240m, net fees) ===
  single-shot (first trig)     mean  +1.29  med  +1.02  win 57.0%  q10  -6.14  m/std +0.197  (n=1727)
  scale-in (ramped down)       mean  +1.82  med  +2.25  win 65.5%  q10  -5.16  m/std +0.313  (n=1727)
  avg tranches/cluster: 2.6 | entry lowered by: med +0.00%  mean +0.62%



=== -7%/15m  (exit = cluster_end+240m, net fees) ===
  single-shot (first trig)     mean  +3.25  med  +3.14  win 63.8%  q10  -5.77  m/std +0.423  (n=776)
  scale-in (ramped down)       mean  +3.42  med  +4.09  win 73.6%  q10  -4.28  m/std +0.561  (n=776)
  avg tranches/cluster: 2.6 | entry lowered by: med +0.00%  mean +0.34%


**Замечание про комиссии (важно, честно):** если общий нотионал = 1 позиция и
мы делим его на N траншей, комиссия берётся с *нотионала каждого филла* — сумма
по входу = тот же FEE, что и у одного входа. То есть «больше филлов» НЕ значит
«больше суммарной комиссии» при фиксированном общем размере. Реальная переплата
от дробления — это **спред/проскальзывание на каждый филл** (тут не
смоделирован) + минимальная комиссия на ордер у некоторых бирж. Поэтому ниже
доп. строка: scale-in с грубым проксИ проскальзывания 2bp на филл.

In [4]:
SLIP=0.0002  # 2bp slippage per fill (rough proxy for splitting the order)
def simulate_slip(thr):
    scaled=[]
    for s in SYMS:
        c=closes[s]; n=len(c)
        for (st,en,tr) in clusters(c,thr):
            ex_idx=en+1+H
            if ex_idx>=n or st+1>=n: continue
            exitp=c[ex_idx]
            ws=np.arange(1,len(tr)+1,dtype=float)
            prices=np.array([c[t+1] for t in tr if t+1<n])
            ws=ws[:len(prices)]; ws/=ws.sum()
            avg_entry=(ws*prices).sum()
            cost=2*FEE + SLIP*len(prices)   # exit+entry fee on notional + per-fill slippage
            scaled.append(exitp/avg_entry-1-cost)
    return np.array(scaled)
for thr in (0.05,0.07):
    a=simulate_slip(thr)
    print(f"-{int(thr*100)}%  scale-in WITH {int(SLIP*1e4)}bp/fill slippage:", end=" ")
    line("",a)

-5%  scale-in WITH 2bp/fill slippage:                                mean  +1.77  med  +2.20  win 65.3%  q10  -5.19  m/std +0.304  (n=1727)


-7%  scale-in WITH 2bp/fill slippage:                                mean  +3.37  med  +4.03  win 73.3%  q10  -4.32  m/std +0.553  (n=776)


## Как читать
- **single-shot vs scale-in** при одинаковом выходе — изолирует эффект усреднения
  входа. Смотри mean, q10 (левый хвост = ножи, что не отскочили) и mean/std.
- **entry lowered by** — насколько ниже средний вход scale-in vs первый триггер.
- Учитывай оговорку про комиссии: дробление НЕ удваивает комиссию (она с
  нотионала); реальный налог — проскальзывание на филл (последняя строка).

*Verdict после прогона.*

## VERDICT (iter 02)

**Scale-in (добор лонга вниз) РАБОТАЕТ — зеркало pump-11 подтверждено.** При
одинаковом выходе, net костов, бьёт single-shot по всем метрикам:
- −7%/15m: single m/std 0.423 → scale-in **0.561**; mean +3.25→+3.42; win
  63.8→73.6%; q10 −5.77→**−4.28** (хвост ЛУЧШЕ).
- −5%/15m: m/std 0.197→**0.313**; mean +1.29→+1.82; win 57→65.5%; q10 −6.14→−5.16.

**Поправка про комиссии:** дробление НЕ удваивает комиссию (она с нотионала,
Σ FEE·вес = FEE). Реальный налог = проскальзывание/филл; прокси 2bp почти не
съедает (avg 2.6 филла): −7% +3.42→+3.37.

**Где выигрыш:** медианный кластер короткий (3–5м) → ~половина = 1 транш
(scale=single). Весь буст от многотраншевых = глубоких/длинных дампов, где рамп
«поздние крупнее» ловит сок отскока.

**Оговорки:** survivorship — улучшенный q10 может быть артефактом (нет умерших
монет; добор в нож в реале рискованнее). 2024/50 syms/overlap. Стоп не нужен.

**Дальше:** OOS 2025/26 (held?), потом survivorship-чек, полная вселенная.